# Programação Orientada a Objetos com Python — Parte 2
## Encapsulamento e Métodos Especiais

**Tema: O Podrão — Sistema de Lanchonete / Food Truck**

**Disciplina:** COMP0496 — Programação A
**Duração:** 200 minutos (Aula 2 de 4 do módulo de POO)

---

### Pré-requisitos

Você deve ter concluído o notebook **Parte 1** e estar confortável com:

- Definir classes com `__init__` e `self`.
- Criar e manipular instâncias.
- Distinguir atributos de classe de atributos de instância.

### Objetivos desta aula

Ao final desta aula, você será capaz de:

1. Explicar o princípio do **encapsulamento** e por que ele importa no design de software.
2. Usar as convenções `_protegido` e `__privado` de Python e entender o mecanismo de *name mangling*.
3. Definir **propriedades** (`@property`) para validação e acesso controlado a atributos.
4. Criar propriedades **somente-leitura** (read-only).
5. Implementar os principais **métodos especiais** (*dunder methods*) — `__str__`, `__repr__`, `__eq__`, `__lt__`, `__len__` — para tornar seus objetos *Pythônicos*.
6. Aplicar tudo isso construindo uma classe `Fracao` completa.

### Como usar este material

Execute cada célula na ordem. Não pule exercícios. Quando ver um bloco ⚠️ **Armadilha**, leia com cuidado — são os erros que todo mundo comete na primeira vez.

---

## 1. Revisão rápida da Aula 1

Antes de avançar, relembre: na aula passada criamos `Lanche`, `Bebida` e `Cardapio`. Cada objeto tinha atributos públicos, acessados diretamente com ponto:

```python
xtudo = Lanche("X-Tudo", 18.50, "pão, hambúrguer, ...")
print(xtudo.preco)     # 18.50
xtudo.preco = -999     # 😱 ninguém está impedindo isso!
```

A última linha é o problema que vamos atacar hoje. **Nada** na nossa classe impede que alguém coloque um preço negativo, uma string no lugar de número, ou sabote o objeto. Isso é o que chamamos de *estado inválido*.

Execute para confirmar:

In [ ]:
class Lanche:
    def __init__(self, nome, preco, ingredientes):
        self.nome = nome
        self.preco = preco
        self.ingredientes = ingredientes

xtudo = Lanche("X-Tudo", 18.50, "pão, hambúrguer, queijo, bacon")
print(f"Preço inicial: R${xtudo.preco:.2f}")

# Sabotagem silenciosa — nenhum erro é gerado
xtudo.preco = -999
xtudo.nome = 42            # nome virou número?!
xtudo.ingredientes = None  # e os ingredientes sumiram

print(f"Preço agora:   R${xtudo.preco}")
print(f"Nome agora:    {xtudo.nome}")
print(f"Ingredientes:  {xtudo.ingredientes}")
print("⚠️  O objeto está em um estado inválido e ninguém percebeu.")


---
## 2. Encapsulamento: ocultando o estado interno

**Encapsulamento** é o princípio de esconder os detalhes internos de um objeto e permitir que seu estado só seja modificado através de uma *interface controlada* — métodos definidos pela própria classe.

### Analogia: a caixa registradora do Podrão

Na caixa registradora física, qualquer cliente vê o **total** na tela (acesso controlado), mas ninguém pode abrir a gaveta e pegar dinheiro diretamente (acesso bloqueado). Os métodos da classe funcionam como esse mecanismo de controle.

```
┌──────────────────────────────────────┐
│        CaixaRegistradora             │
│                                      │
│  ┌────────────────────────────────┐  │
│  │  __saldo  (privado, oculto)    │  │  ← ninguém mexe direto
│  └──────────────┬─────────────────┘  │
│                 │                    │
│  ┌──────────────▼─────────────────┐  │
│  │ receber_pagamento(valor)       │  │  ← interface pública
│  │ realizar_despesa(valor)        │  │     (única forma de acesso)
│  │ ver_saldo()                    │  │
│  └────────────────────────────────┘  │
└──────────────────────────────────────┘
```

### 2.1 As convenções de Python: `_protegido` e `__privado`

Python **não tem** modificadores de acesso como `private` e `public` do Java/C++. Em vez disso, usa **convenções**:

| Nome | Significado | Quem respeita |
|---|---|---|
| `nome` | público — uso livre | todos |
| `_nome` | "protegido" — uso interno, não mexa | convenção social |
| `__nome` | "privado" — com *name mangling* | mecanismo real do Python |

A citação clássica da comunidade Python é:

> *"We're all consenting adults here."*

Ou seja: um `_atributo` é uma placa dizendo "não mexa"; você *pode* ignorar, mas a responsabilidade é sua. Já o `__atributo` (duplo underscore) ativa um mecanismo chamado ***name mangling***, que renomeia o atributo internamente para dificultar (mas não impedir totalmente) o acesso externo.

In [ ]:
class CaixaRegistradora:
    """
    Controla o fluxo financeiro do food truck.
    O saldo é privado: só pode ser alterado pelos métodos da classe.
    """

    def __init__(self, saldo_inicial=0.0):
        self.__saldo = saldo_inicial     # ← atributo privado (prefixo '__')
        self.__historico = []            # ← também privado

    def receber_pagamento(self, valor):
        """Adiciona valor ao saldo, com validação."""
        if valor <= 0:
            print("  ✘ Valor inválido. Pagamento rejeitado.")
            return
        self.__saldo += valor
        self.__historico.append(f"+ R${valor:.2f}")
        print(f"  ✔ Pagamento de R${valor:.2f} recebido.")

    def realizar_despesa(self, valor):
        """Desconta valor do saldo, com verificação de fundos."""
        if valor > self.__saldo:
            print("  ✘ Saldo insuficiente! Não dá pra pagar.")
            return
        self.__saldo -= valor
        self.__historico.append(f"- R${valor:.2f}")
        print(f"  ✔ Despesa de R${valor:.2f} registrada.")

    def ver_saldo(self):
        """Única forma legítima de consultar o saldo de fora."""
        print(f"  💰 Saldo atual: R${self.__saldo:.2f}")

    def extrato(self):
        """Mostra todas as transações do turno."""
        print("  📋 Extrato do turno:")
        for i, t in enumerate(self.__historico, 1):
            print(f"     {i}. {t}")


caixa = CaixaRegistradora(50.00)
caixa.ver_saldo()
caixa.receber_pagamento(18.50)   # X-Tudo
caixa.receber_pagamento(12.00)   # Dogão
caixa.realizar_despesa(10.00)    # Comprou gás
caixa.realizar_despesa(200.00)   # Deve ser bloqueado
caixa.ver_saldo()
caixa.extrato()


Agora vamos tentar **sabotar** o objeto acessando `__saldo` diretamente:

In [ ]:
# Tentativa de sabotagem — execute e observe o erro
try:
    print(caixa.__saldo)
except AttributeError as e:
    print(f"AttributeError: {e}")
    print("✔ Encapsulamento funcionou: o atributo não está acessível diretamente.")


### ⚠️ Armadilha: name mangling não é "segurança de verdade"

O Python, nos bastidores, apenas **renomeia** `__saldo` para `_CaixaRegistradora__saldo`. Ou seja, se alguém realmente quiser, ainda consegue acessar — como veremos abaixo. O objetivo do `__` não é segurança, é **evitar colisões** em hierarquias de herança e **sinalizar** fortemente que o atributo é interno.

In [ ]:
# O que realmente acontece: Python renomeia o atributo
print("Atributos reais do objeto:")
for attr in vars(caixa):
    print(f"  {attr}")

# E é possível acessar pelo nome mangled (mas não faça isso!)
print(f"\nAcesso via name mangling: R${caixa._CaixaRegistradora__saldo:.2f}")
print("⚠️  Funciona, mas viola o contrato da classe. Não faça em código real.")


### Exercício 2.1
Crie uma classe `Estoque` para o food truck com:

- Um atributo privado `__itens` (dicionário `{nome: quantidade}`), iniciado vazio.
- Método `adicionar(nome, quantidade)` que soma ao estoque (cria a entrada se não existir).
- Método `retirar(nome, quantidade)` que verifica se há quantidade suficiente antes de retirar; se não houver, imprime `"✘ Estoque insuficiente de <nome>"`.
- Método `mostrar()` que imprime todos os itens e quantidades.

Teste adicionando e retirando pães, salsichas e queijos.

In [ ]:
# Escreva sua solução aqui



---
## 3. `@property`: acesso controlado de forma Pythônica

O jeito de `CaixaRegistradora` funciona, mas tem um incômodo: para **ler** o saldo eu preciso chamar `caixa.ver_saldo()` como método — não posso simplesmente fazer `caixa.saldo`.

Em linguagens como Java a solução é criar um monte de `getSaldo()` e `setSaldo(valor)`. Funciona, mas é verboso. Python oferece uma alternativa muito mais elegante: o decorador **`@property`**, que permite que um método seja *acessado como se fosse um atributo*.

### 3.1 A ideia básica

In [ ]:
class Lanche:
    """Lanche com preço validado via @property."""

    def __init__(self, nome, preco, ingredientes):
        self.nome = nome
        self._preco = preco                 # atributo "real" (protegido)
        self.ingredientes = ingredientes

    @property
    def preco(self):
        """Getter: chamado quando alguém faz `lanche.preco`."""
        return self._preco

    @preco.setter
    def preco(self, novo_valor):
        """Setter: chamado quando alguém faz `lanche.preco = X`."""
        if not isinstance(novo_valor, (int, float)):
            raise TypeError("Preço deve ser numérico.")
        if novo_valor < 0:
            raise ValueError("Preço não pode ser negativo.")
        if novo_valor > 500:
            raise ValueError("Preço suspeito — está vendendo lanche ou carro?")
        self._preco = novo_valor


xtudo = Lanche("X-Tudo", 18.50, "pão, hambúrguer, queijo, bacon")
print(f"Preço: R${xtudo.preco:.2f}")   # chama o getter — parece atributo!

xtudo.preco = 19.00                    # chama o setter — validação passa
print(f"Novo preço: R${xtudo.preco:.2f}")

# Tentativa inválida
try:
    xtudo.preco = -5
except ValueError as e:
    print(f"✔ Bloqueado: {e}")

try:
    xtudo.preco = "caro"
except TypeError as e:
    print(f"✔ Bloqueado: {e}")


**Repare no detalhe importante:** quem usa a classe escreve `xtudo.preco` e `xtudo.preco = 19.00` — a sintaxe é idêntica à de um atributo comum. Mas por baixo dos panos, cada acesso passa pelo seu código de validação. Isso é o que os Pythonistas chamam de *"uniform access principle"*.

### 3.2 Propriedades somente-leitura (*read-only*)

Se você definir **apenas** o getter (sem `@preco.setter`), a propriedade se torna somente-leitura: tentar atribuir um valor gera `AttributeError`. Útil para valores *derivados*.

In [ ]:
class Pedido:
    """Pedido com total calculado automaticamente."""

    def __init__(self, cliente):
        self.cliente = cliente
        self._itens = []          # lista de tuplas (nome, preco)

    def adicionar(self, nome, preco):
        self._itens.append((nome, preco))

    @property
    def total(self):
        """Somente-leitura: sempre calculado a partir dos itens."""
        return sum(preco for _, preco in self._itens)

    @property
    def quantidade(self):
        """Somente-leitura: número de itens no pedido."""
        return len(self._itens)


p = Pedido("João")
p.adicionar("X-Tudo", 18.50)
p.adicionar("Dogão", 12.00)
p.adicionar("Cajuína", 5.00)

print(f"Cliente: {p.cliente}")
print(f"Itens: {p.quantidade}")        # sem parênteses — parece atributo
print(f"Total: R${p.total:.2f}")       # idem

# Tentar atribuir vai falhar
try:
    p.total = 0   # cliente tentando fraudar o sistema 😈
except AttributeError as e:
    print(f"✔ Read-only bloqueou: {e}")


### ⚠️ Armadilha: não confunda o atributo "real" com a propriedade

No exemplo do `Lanche`, o atributo interno se chama `_preco` e a propriedade se chama `preco`. **Não** use o mesmo nome, ou você cria recursão infinita:

```python
# ❌ ERRADO — causa RecursionError
@property
def preco(self):
    return self.preco   # chama a própria propriedade de novo!

@preco.setter
def preco(self, v):
    self.preco = v      # idem, loop infinito
```

A convenção é: propriedade pública `preco`, atributo interno `_preco`.

### Exercício 3.1
Crie uma classe `Termometro` que representa a temperatura do freezer do food truck:

- No `__init__`, receba uma temperatura em Celsius.
- Use `@property` para `celsius` com um setter que valida: a temperatura não pode ser menor que -50 nem maior que 60 (senão o sensor está quebrado).
- Crie uma propriedade **somente-leitura** `fahrenheit` que retorna a temperatura convertida (`F = C * 9/5 + 32`).

Teste criando um termômetro, mudando a temperatura, e tentando valores inválidos.

In [ ]:
# Escreva sua solução aqui



### Exercício 3.2
Volte à sua classe `Estoque` do Exercício 2.1 e transforme `__itens` em uma propriedade somente-leitura chamada `itens`, para que o usuário possa **ler** o dicionário mas não substituí-lo (pense no que acontece se ele fizer `estoque.itens = {}` sem querer).

In [ ]:
# Escreva sua solução aqui



---
## ☕ Intervalo (15 min)

Até aqui vimos: encapsulamento, `_protegido`, `__privado`, `@property`, setters com validação e propriedades read-only. Depois do intervalo vamos para a outra metade da aula: **métodos especiais** (dunders).

---

## 4. Métodos especiais (*dunder methods*)

Você já conhece vários métodos começando e terminando com duplo underscore: `__init__`, `__str__`, etc. Eles são chamados de **métodos especiais** ou, informalmente, ***dunders*** (de "*double underscore*"). Python chama esses métodos **automaticamente** em situações específicas.

Por exemplo, quando você escreve `print(objeto)`, Python internamente chama `objeto.__str__()`. Quando escreve `a == b`, chama `a.__eq__(b)`. Quando faz `len(lista)`, chama `lista.__len__()`.

Isso significa que, implementando os dunders certos, **suas classes passam a se comportar como as classes internas do Python** — é o que torna um objeto "Pythônico".

### 4.1 `__str__` e `__repr__`: representação textual

Observe o que acontece quando a gente tenta imprimir um objeto sem `__str__`:

In [ ]:
class LancheRuim:
    def __init__(self, nome, preco):
        self.nome = nome
        self.preco = preco

x = LancheRuim("X-Tudo", 18.50)
print(x)                  # saída inútil
print(repr(x))            # também inútil
print([x, x])             # pior ainda dentro de uma lista


Agora vamos implementar `__str__` (representação amigável, para usuários) e `__repr__` (representação técnica, para desenvolvedores — idealmente deveria poder ser copiada e colada para recriar o objeto).

In [ ]:
class Lanche:
    def __init__(self, nome, preco, ingredientes):
        self.nome = nome
        self.preco = preco
        self.ingredientes = ingredientes

    def __str__(self):
        """Representação amigável — chamada por print() e str()."""
        return f"🍔 {self.nome} (R${self.preco:.2f})"

    def __repr__(self):
        """Representação técnica — chamada no console, em listas, por repr()."""
        return f"Lanche(nome={self.nome!r}, preco={self.preco!r}, ingredientes={self.ingredientes!r})"


xtudo = Lanche("X-Tudo", 18.50, "pão, hambúrguer, queijo, bacon")
dogao = Lanche("Dogão", 12.00, "pão, salsicha, vinagrete")

print("Com print (usa __str__):")
print(xtudo)
print(dogao)

print("\nCom repr (usa __repr__):")
print(repr(xtudo))

print("\nEm uma lista (usa __repr__ de cada elemento):")
print([xtudo, dogao])


**Regra prática:**

- `__str__` → "bonito", para usuários finais. Se não definir, Python usa `__repr__` como fallback.
- `__repr__` → "inequívoco", para programadores. Idealmente `eval(repr(obj))` recriaria o objeto.
- Se só for implementar um dos dois, implemente `__repr__`: ele serve como fallback do `__str__` e aparece no console, em listas, em mensagens de erro.

Note o uso de `{self.nome!r}` na f-string: o `!r` aplica `repr()` ao valor, adicionando aspas em strings automaticamente.

### 4.2 `__eq__` e `__lt__`: comparando objetos

Por padrão, dois objetos só são iguais (`==`) se forem **o mesmo objeto em memória** — mesmo que tenham atributos idênticos. Veja:

In [ ]:
a = Lanche("X-Tudo", 18.50, "pão, hambúrguer, queijo, bacon")
b = Lanche("X-Tudo", 18.50, "pão, hambúrguer, queijo, bacon")

print(f"a == b?  {a == b}")   # False — são objetos diferentes!
print(f"a is b?  {a is b}")   # False — idem
print("Mas eles têm os mesmos atributos... Esperávamos True em '==' ")


A solução é implementar `__eq__`, que define o que significa "igual" para sua classe. E, aproveitando, vamos implementar `__lt__` (*less than*) para permitir comparação e ordenação por preço.

In [ ]:
class Lanche:
    def __init__(self, nome, preco, ingredientes):
        self.nome = nome
        self.preco = preco
        self.ingredientes = ingredientes

    def __str__(self):
        return f"🍔 {self.nome} (R${self.preco:.2f})"

    def __repr__(self):
        return f"Lanche({self.nome!r}, {self.preco!r})"

    def __eq__(self, outro):
        """Dois lanches são iguais se tiverem mesmo nome e preço."""
        if not isinstance(outro, Lanche):
            return NotImplemented       # permite comparação simétrica com outros tipos
        return self.nome == outro.nome and self.preco == outro.preco

    def __lt__(self, outro):
        """Ordenação: lanches mais baratos vêm antes."""
        if not isinstance(outro, Lanche):
            return NotImplemented
        return self.preco < outro.preco


a = Lanche("X-Tudo", 18.50, "pão, hambúrguer, queijo, bacon")
b = Lanche("X-Tudo", 18.50, "pão, hambúrguer, queijo, bacon")
c = Lanche("Dogão", 12.00, "pão, salsicha, vinagrete")

print(f"a == b?  {a == b}")   # True — mesmo nome e preço
print(f"a == c?  {a == c}")   # False
print(f"c < a?   {c < a}")    # True — Dogão é mais barato

# E ganhamos sort() de graça, porque sort usa __lt__
cardapio = [a, c, Lanche("Açaí", 15.00, "açaí, granola, banana")]
cardapio_ordenado = sorted(cardapio)
print("\nOrdenado por preço:")
for item in cardapio_ordenado:
    print(f"  {item}")


**Detalhes importantes:**

- Retornar `NotImplemented` (não é o mesmo que `NotImplementedError`) quando o outro operando é de tipo incompatível permite que Python tente a comparação no sentido inverso e, se falhar, retorne `False` educadamente.
- Implementar `__lt__` já é suficiente para `sorted()` funcionar. Se quiser todas as comparações (`<=`, `>`, `>=`), implemente as outras ou use `@functools.total_ordering`.

### ⚠️ Armadilha: `__eq__` e `__hash__` andam juntos

Quando você define `__eq__`, o Python **automaticamente** define `__hash__ = None`, tornando seus objetos **não-hasháveis** (não podem ser usados em `set` nem como chave de `dict`). Veja:

In [ ]:
try:
    s = {a, b, c}       # tentar criar um conjunto de lanches
except TypeError as e:
    print(f"✘ Erro: {e}")
    print("✔ Isso acontece porque definimos __eq__ mas não __hash__.")


Se fizer sentido que seus objetos sejam hasháveis (imutáveis e comparáveis por valor), implemente também `__hash__`:

In [ ]:
class LancheHashavel:
    def __init__(self, nome, preco):
        self.nome = nome
        self.preco = preco

    def __eq__(self, outro):
        if not isinstance(outro, LancheHashavel):
            return NotImplemented
        return self.nome == outro.nome and self.preco == outro.preco

    def __hash__(self):
        # Hash baseado na mesma chave usada por __eq__
        return hash((self.nome, self.preco))

    def __repr__(self):
        return f"LancheHashavel({self.nome!r}, {self.preco!r})"


x1 = LancheHashavel("X-Tudo", 18.50)
x2 = LancheHashavel("X-Tudo", 18.50)
d  = LancheHashavel("Dogão", 12.00)

conjunto = {x1, x2, d}
print(f"Conjunto tem {len(conjunto)} elementos (esperado: 2, pois x1 == x2)")
print(conjunto)


### 4.3 `__len__`: tamanho de uma coleção

Se sua classe representa algo que tem "tamanho" (uma coleção, um container), implementar `__len__` permite usar a função built-in `len()`:

In [ ]:
class Pedido:
    def __init__(self, cliente):
        self.cliente = cliente
        self.itens = []

    def adicionar(self, lanche):
        self.itens.append(lanche)

    def __len__(self):
        """Tamanho do pedido = número de itens."""
        return len(self.itens)

    def __str__(self):
        return f"Pedido de {self.cliente} com {len(self)} iten(s)"


p = Pedido("Maria")
p.adicionar(Lanche("X-Tudo", 18.50, "..."))
p.adicionar(Lanche("Dogão", 12.00, "..."))
p.adicionar(Lanche("Açaí", 15.00, "..."))

print(p)
print(f"len(p) = {len(p)}")   # chama p.__len__()
print(f"Pedido vazio? {len(p) == 0}")


### Tabela-resumo dos principais dunders

| Dunder | Chamado por | Propósito |
|---|---|---|
| `__init__(self, ...)` | `Classe(...)` | Construtor |
| `__str__(self)` | `print(obj)`, `str(obj)` | Representação amigável |
| `__repr__(self)` | console, listas, `repr(obj)` | Representação técnica |
| `__eq__(self, outro)` | `a == b` | Igualdade por valor |
| `__lt__(self, outro)` | `a < b`, `sorted(...)` | Ordenação |
| `__hash__(self)` | `hash(obj)`, `set`, `dict` | Hashabilidade |
| `__len__(self)` | `len(obj)` | Tamanho |
| `__contains__(self, x)` | `x in obj` | Pertencimento |
| `__getitem__(self, i)` | `obj[i]` | Indexação |
| `__add__(self, outro)` | `a + b` | Adição |

Vamos usar vários deles na prática guiada a seguir.

---

## 5. Prática guiada: classe `Fracao`

Agora vamos construir a classe que mais exercita dunders: uma `Fracao` (número racional) com validação, representação, igualdade, ordenação e operações aritméticas.

Esse é um exercício **clássico** de POO. Estude o código, execute, modifique.

In [ ]:
from math import gcd   # máximo divisor comum


class Fracao:
    """Número racional representado como numerador/denominador."""

    def __init__(self, numerador, denominador=1):
        if denominador == 0:
            raise ValueError("Denominador não pode ser zero.")
        # Simplifica já na construção
        divisor = gcd(abs(numerador), abs(denominador))
        # Convenção: sinal fica no numerador
        sinal = -1 if (numerador * denominador) < 0 else 1
        self._num = sinal * abs(numerador) // divisor
        self._den = abs(denominador) // divisor

    # ───── Propriedades read-only ─────
    @property
    def numerador(self):
        return self._num

    @property
    def denominador(self):
        return self._den

    @property
    def valor(self):
        """Valor decimal (read-only, derivado)."""
        return self._num / self._den

    # ───── Representação ─────
    def __str__(self):
        if self._den == 1:
            return str(self._num)
        return f"{self._num}/{self._den}"

    def __repr__(self):
        return f"Fracao({self._num}, {self._den})"

    # ───── Comparações ─────
    def __eq__(self, outro):
        if not isinstance(outro, Fracao):
            return NotImplemented
        # Como simplificamos no __init__, basta comparar os componentes
        return self._num == outro._num and self._den == outro._den

    def __lt__(self, outro):
        if not isinstance(outro, Fracao):
            return NotImplemented
        # a/b < c/d  ⟺  a*d < c*b  (válido pois denominadores são positivos)
        return self._num * outro._den < outro._num * self._den

    def __hash__(self):
        return hash((self._num, self._den))

    # ───── Aritmética ─────
    def __add__(self, outro):
        if not isinstance(outro, Fracao):
            return NotImplemented
        # a/b + c/d = (a*d + c*b) / (b*d)
        novo_num = self._num * outro._den + outro._num * self._den
        novo_den = self._den * outro._den
        return Fracao(novo_num, novo_den)

    def __sub__(self, outro):
        if not isinstance(outro, Fracao):
            return NotImplemented
        novo_num = self._num * outro._den - outro._num * self._den
        novo_den = self._den * outro._den
        return Fracao(novo_num, novo_den)

    def __mul__(self, outro):
        if not isinstance(outro, Fracao):
            return NotImplemented
        return Fracao(self._num * outro._num, self._den * outro._den)

    def __neg__(self):
        """Unário: -fracao"""
        return Fracao(-self._num, self._den)


# ───── Demonstração ─────
a = Fracao(1, 2)
b = Fracao(1, 3)
c = Fracao(2, 4)   # deve simplificar para 1/2

print(f"a = {a}")
print(f"b = {b}")
print(f"c = {c}                 (simplificada automaticamente)")

print(f"\na == c?  {a == c}    (iguais após simplificação)")
print(f"a < b?   {a < b}")
print(f"a > b?   {a > b}         (nota: funciona mesmo sem __gt__, Python deriva)")

print(f"\na + b = {a + b}")
print(f"a - b = {a - b}")
print(f"a * b = {a * b}")
print(f"-a    = {-a}")

print(f"\nValor decimal de a: {a.valor}")
print(f"Numerador de (a+b): {(a+b).numerador}")

# E como implementamos __hash__, podemos usar em conjuntos
fracoes = {Fracao(1,2), Fracao(2,4), Fracao(1,3), Fracao(3,9)}
print(f"\nConjunto simplificado: {fracoes}  (duplicatas removidas pelo set)")

# E ordenar
print(f"Ordenadas: {sorted([Fracao(3,4), Fracao(1,2), Fracao(5,6), Fracao(1,3)])}")


Observe o que essa classe demonstra de uma só vez:

1. **Validação no construtor** (denominador não-zero).
2. **Simplificação automática** no `__init__` — garante que `2/4` e `1/2` sejam realmente iguais.
3. **Propriedades read-only** (`numerador`, `denominador`, `valor`).
4. **Representação dupla** (`__str__` vs `__repr__`).
5. **Igualdade e ordenação** coerentes (`__eq__`, `__lt__`, `__hash__`).
6. **Aritmética natural** com `__add__`, `__sub__`, `__mul__`, `__neg__`.

Tudo isso permite que o usuário da classe escreva `a + b`, `sorted(fracoes)`, `Fracao(1,2) in conjunto` — **exatamente como se `Fracao` fosse um tipo interno do Python**. É esse o objetivo dos dunders.

---
## 6. Exercícios de consolidação

### Exercício 6.1 — Completar a `Fracao`
Adicione à classe `Fracao`:

1. O método `__truediv__` para permitir `a / b` (divisão de frações: `(a/b) / (c/d) = (a*d)/(b*c)`). Levante `ZeroDivisionError` se o numerador do divisor for zero.
2. O método `__abs__` para que `abs(Fracao(-3, 4))` retorne `Fracao(3, 4)`.
3. O método `__bool__` para que `bool(Fracao(0))` seja `False` e qualquer outra fração seja `True`.

Teste cada adição.

In [ ]:
# Escreva sua solução aqui (copie a Fracao e estenda)



### Exercício 6.2 — Classe `Temperatura`
Modele uma classe `Temperatura` imutável que representa uma temperatura internamente em Kelvin:

- Construtor recebe um valor em Celsius.
- Propriedades read-only `celsius`, `fahrenheit` e `kelvin`.
- Validação: não permita temperaturas abaixo do zero absoluto (-273.15 °C).
- `__str__` retorna, por exemplo, `"25.0°C"`.
- `__repr__` retorna `"Temperatura(25.0)"`.
- `__eq__` e `__lt__` comparam duas temperaturas pelo valor em Kelvin.
- `__hash__` para que seja usável em `set`.

Teste criando várias temperaturas, ordenando uma lista delas e colocando num conjunto.

In [ ]:
# Escreva sua solução aqui



### Exercício 6.3 — Integrador (para casa): `ContaBancaria`
Modele uma classe `ContaBancaria` que combina **tudo** da Aula 1 e da Aula 2:

- Atributos privados `__saldo` e `__historico` (lista de transações).
- Atributo de classe `taxa_saque = 0.50` (taxa fixa por saque).
- Propriedade read-only `saldo`.
- Propriedade `titular` com setter que rejeita nomes vazios ou com menos de 3 caracteres.
- Métodos `depositar(valor)` e `sacar(valor)` com validação (valor positivo, saldo suficiente, desconta a taxa do saque).
- `__str__` retornando `"Conta de <titular>: R$<saldo>"`.
- `__repr__` técnico.
- `__eq__` comparando contas pelo número (adicione um atributo `numero` no construtor).
- `__len__` retornando o número de transações no histórico.



In [ ]:
# Escreva sua solução aqui (tarefa para casa)



---
## 7. Resumo da aula

| Conceito | Definição | Exemplo |
|---|---|---|
| **Encapsulamento** | Ocultar estado interno; acesso via interface controlada | `self.__saldo` |
| **`_protegido`** | Convenção: "uso interno, não mexa" (mas nada impede) | `self._preco` |
| **`__privado`** | *Name mangling*: renomeado internamente | `self.__saldo` → `_Classe__saldo` |
| **`@property`** | Define getter que se usa como atributo | `lanche.preco` (parece atributo) |
| **`@x.setter`** | Define setter com validação | `lanche.preco = 19.00` |
| **Propriedade read-only** | Getter sem setter — só leitura | `pedido.total` (derivado) |
| **`__str__`** | Representação amigável (para usuário) | `print(obj)` |
| **`__repr__`** | Representação técnica (para dev) | console, listas |
| **`__eq__`** | Igualdade por valor | `a == b` |
| **`__lt__`** | Ordenação; habilita `sorted()` | `a < b` |
| **`__hash__`** | Torna objeto hashável (set, dict key) | `hash(obj)` |
| **`__len__`** | Tamanho da coleção | `len(obj)` |
| **`__add__`, `__sub__`, `__mul__`** | Operadores aritméticos | `a + b` |

### Armadilhas a lembrar

1. `__` (duplo underscore) **não é segurança**, é *name mangling*. Serve para evitar colisões e sinalizar forte intenção de privacidade.
2. **Nunca** dê o mesmo nome à propriedade e ao atributo interno — causa recursão infinita. Convenção: `preco` (propriedade) e `_preco` (interno).
3. Ao definir `__eq__`, **sempre considere** se precisa também de `__hash__`. Se o objeto é mutável, é mais seguro deixar `__hash__` desabilitado.
4. Retorne `NotImplemented` (e não `False`) em dunders de comparação quando o outro operando é de tipo incompatível. Isso permite que o Python tente o sentido inverso.
5. `__str__` é para humanos, `__repr__` é para desenvolvedores. Se só for implementar um, prefira `__repr__`.

### O que vem na Aula 3

Na próxima aula vamos cobrir os mecanismos de **reutilização e extensão de código**:

- **Herança** simples e múltipla (`class Filha(Pai)`, `super()`, MRO).
- **Polimorfismo** via *duck typing* e sobrescrita de métodos.
- **Composição** vs herança — quando usar cada uma.
- **Classes abstratas** (`abc.ABC`, `@abstractmethod`).

Até lá!